# CT-Restore: Colab / RunPod end-to-end

This notebook installs the repository, detects CUDA, downloads a guarded TCIA sample, preprocesses DICOM to HU-preserving NIfTI, runs one smoke-training epoch, and performs inference. **Research use only.** The smoke path bypasses manual visual approval but not automatic QC; its weights must never be used clinically.

In [ ]:
# Edit REPO_URL after publishing/forking this repository. Leave blank if already cloned.
REPO_URL = "https://github.com/godsonj64/RecoverCT.git"
BRANCH = "main"
USE_GOOGLE_DRIVE = False  # Persistent but slower than Colab local SSD.
DOWNLOAD_LIMIT = 1         # Keep 1 for the smoke run; 0 means the full collection.
EPOCHS = 1
ACCEPT_TCIA_DATA_TERMS = False  # Set True only after reviewing TCIA terms/citation.

# Collection used for the SMOKE RUN only. This must be one the public NBIA API
# actually serves. The head-and-neck planning collections this project targets
# (HNC-IMRT-70-33, HEAD-NECK-PET-CT) are NOT reachable anonymously -- they need an
# NBIA login or the official Data Retriever, so they cannot be used here.
# Pancreas-CT is abdominal diagnostic CT: it exercises the pipeline, not the science.
COLLECTION = "Pancreas-CT"


## Optional persistent Google Drive
RunPod users should attach a sufficiently large volume at `/workspace`. Colab users can mount Drive for checkpoints, but local `/content` storage is faster and disappears when the runtime ends.

In [ ]:
from pathlib import Path
import os, subprocess, sys

IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ or 'google.colab' in sys.modules
IN_RUNPOD = 'RUNPOD_POD_ID' in os.environ
if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError('Google Drive mounting is only available in Colab')
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/ct_restore_data')
elif IN_RUNPOD:
    DATA_ROOT = Path('/workspace/ct_restore_data')
elif IN_COLAB:
    DATA_ROOT = Path('/content/ct_restore_data')
else:
    DATA_ROOT = Path.cwd() / 'data'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['CT_RESTORE_DATA_ROOT'] = str(DATA_ROOT)
print({'colab': IN_COLAB, 'runpod': IN_RUNPOD, 'data_root': str(DATA_ROOT)})


## Clone and install
The bootstrap keeps the runtime's CUDA-compatible PyTorch when already installed and installs the repository in editable mode.

In [ ]:
candidate = Path.cwd()
if not (candidate / 'pyproject.toml').exists():
    if not REPO_URL:
        raise ValueError('Set REPO_URL above, or run this notebook from an existing clone')
    repo_dir = Path('/workspace/ct-restore' if IN_RUNPOD else '/content/ct-restore')
    if not repo_dir.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
else:
    repo_dir = candidate
subprocess.run(['bash', 'scripts/bootstrap_notebook.sh'], check=True, env=os.environ.copy())
print('Repository:', Path.cwd())


## Hardware report and resolved configuration
CUDA is selected automatically. BF16 is used on supported Ampere-or-newer GPUs; otherwise FP16 is used. The configuration scales patch size conservatively with VRAM and retains CPU/MPS fallbacks.

In [ ]:
subprocess.run(['ct-restore', 'doctor'], check=True)
resolved = DATA_ROOT / 'configs' / 'notebook_hardware.yaml'
subprocess.run(['ct-restore', 'configure', 'configs/notebook.yaml', str(resolved)], check=True)


## TCIA terms and complete smoke pipeline
Review the [TCIA data usage policy](https://www.cancerimagingarchive.net/data-usage-policies-and-restrictions/) and the citation requirements for whichever collection `COLLECTION` names. The command will not download unless the explicit toggle is enabled. At least 10 GB free is required even for the sample; full training needs substantially more persistent storage.

**Smoke run only.** `COLLECTION` defaults to a collection the anonymous NBIA API serves, so this notebook verifies the pipeline end to end. It does **not** train a head-and-neck model. The target collections ([HNC-IMRT-70-33](https://www.cancerimagingarchive.net/collection/hnc-imrt-70-33/), HEAD-NECK-PET-CT) return zero series without authenticated access; fetch them with the official Data Retriever and point `--data-root` at the result.


In [ ]:
if not ACCEPT_TCIA_DATA_TERMS:
    raise ValueError('Review the linked TCIA terms, then set ACCEPT_TCIA_DATA_TERMS=True')
command = [
    sys.executable, 'scripts/run_notebook_pipeline.py',
    '--data-root', str(DATA_ROOT),
    '--collection', COLLECTION,
    '--limit', str(DOWNLOAD_LIMIT),
    '--epochs', str(EPOCHS),
    '--accept-data-terms',
]
subprocess.run(command, check=True, env=os.environ.copy())


## Inspect research output
The uncertainty map is uncalibrated. It is displayed only to verify that the pipeline completed.

In [ ]:
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

restored_path = DATA_ROOT / 'outputs' / 'notebook_restored.nii.gz'
uncertainty_path = DATA_ROOT / 'outputs' / 'notebook_restored_uncertainty.nii.gz'
restored = np.asarray(nib.load(restored_path).dataobj)
uncertainty = np.asarray(nib.load(uncertainty_path).dataobj)
z = restored.shape[2] // 2
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(restored[:, :, z].T, cmap='gray', vmin=-200, vmax=300, origin='lower')
axes[0].set_title('Restored CT (HU window)')
axes[1].imshow(uncertainty[:, :, z].T, cmap='magma', origin='lower')
axes[1].set_title('Uncalibrated uncertainty')
for axis in axes: axis.axis('off')
plt.show()


## Moving from smoke run to a real experiment
1. Use persistent storage (RunPod `/workspace` volume or Drive/cloud storage). 2. Set `DOWNLOAD_LIMIT=0` only with enough space. 3. Run preprocessing, visually inspect every accepted target, and set `manual_approved=true` in the manifest. 4. Train `configs/pretrain.yaml`, then `configs/finetune.yaml`; use `configs/paired_real.yaml` only for reviewed registered pairs. 5. Perform the full clinical validation protocol in `docs/clinical_validation.md`. Do not promote smoke-run weights.